In [1]:
import sys

sys.path.append("../src")

import altair as alt
alt.renderers.enable("jupyter", offline=True)
alt.data_transformers.disable_max_rows()

import gc
import os
import polars as pl
import polars.selectors as cs
import warnings
from sklearn.model_selection import GroupKFold
from dataclasses import asdict

import lightgbm as lgb
from dotenv import load_dotenv

import wandb
from pathlib import Path

from config import cfg
from data.data_class import Dfs
from data.data_process import add_fold, get_Xy
from data.simple_feature_eng import preprocess
from models.lgb import train_model

# warnings.filterwarnings("ignore")
# warnings.simplefilter("ignore")

cfg.train_path = Path("../data/train.csv")
cfg.test_path = Path("../data/test.csv")
cfg.pltpd_path = Path("../data/podcast_dataset.csv")


df_test = pl.read_csv(cfg.test_path)

df_train = pl.read_csv(cfg.train_path)
df_train = df_train.filter(pl.col("Number_of_Ads").is_not_null())

# df_train = df_train.drop("id")
df_train = add_fold(df_train)
df_train = preprocess(df_train)

df_pltpd = pl.read_csv(cfg.pltpd_path)
df_pltpd = df_pltpd.filter(pl.col("Episode_Length_minutes").is_not_null())
df_pltpd = df_pltpd.with_columns(pl.col("Number_of_Ads").cast(pl.Float64))
df_pltpd = add_fold(df_pltpd)
df_pltpd = preprocess(df_pltpd)
df_pltpd = df_pltpd.with_columns(pl.Series(range(1_000_000, 1_000_000 + len(df_pltpd))).alias("id"))
# df_pltpd

df = df_train.clone()
df

id,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,fold,Episode_Num,Episode_Length_minutes_NaN,Guest_Popularity_percentage_NaN,Episode_Num_Cat
i64,str,f64,str,f64,str,str,f64,f64,str,f64,cat,i32,cat,cat,cat
0,"""0""",63.84,"""0""",74.81,"""3""","""21""",53.58,0.0,"""2""",31.41998,"""Mystery Matters_Episode 98_74.…",98,"""true""","""true""","""98"""
1,"""1""",119.8,"""1""",66.95,"""5""","""14""",75.95,2.0,"""0""",88.01241,"""Joke Junction_Episode 26_66.95…",26,"""false""","""false""","""26"""
2,"""2""",73.9,"""2""",69.97,"""1""","""17""",8.97,0.0,"""0""",44.92531,"""Study Sessions_Episode 16_69.9…",16,"""false""","""false""","""16"""
3,"""3""",67.17,"""3""",57.22,"""0""","""10""",78.7,2.0,"""2""",46.27824,"""Digital Digest_Episode 45_57.2…",45,"""false""","""false""","""45"""
4,"""4""",110.51,"""4""",80.07,"""0""","""14""",58.68,3.0,"""1""",75.61031,"""Mind & Body_Episode 86_80.07_M…",86,"""false""","""false""","""86"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
749995,"""36""",75.66,"""2""",69.36,"""5""","""10""",53.58,0.0,"""0""",56.87058,"""Learning Lab_Episode 25_69.36_…",25,"""false""","""true""","""25"""
749996,"""19""",75.75,"""8""",35.21,"""5""","""21""",53.58,2.0,"""1""",45.46242,"""Business Briefs_Episode 21_35.…",21,"""false""","""true""","""21"""
749997,"""37""",30.98,"""9""",78.58,"""3""","""10""",84.89,0.0,"""0""",15.26,"""Lifestyle Lounge_Episode 51_78…",51,"""false""","""false""","""51"""


In [2]:
import numpy as np

def calculate_rmse(actual, predicted):
    squared_diff = (actual - predicted) ** 2
    mean_squared_diff = squared_diff.mean()
    rmse = np.sqrt(mean_squared_diff)
    return rmse

def optimize_scaling_factor(input_data, target_data, n_searches=20):
    initial_x = target_data.mean() / input_data.mean()
    
    # Define search range
    lower_bound = initial_x * 0.5
    upper_bound = initial_x * 1.5
    
    # Perform n binary searches
    for _ in range(n_searches):
        mid_point = (lower_bound + upper_bound) / 2
        
        delta = (upper_bound - lower_bound) * 0.1
        
        lower_x = mid_point - delta
        upper_x = mid_point + delta
        
        lower_rmse = calculate_rmse(target_data, input_data * lower_x)
        upper_rmse = calculate_rmse(target_data, input_data * upper_x)
        
        if lower_rmse < upper_rmse:
            upper_bound = mid_point
        else:
            lower_bound = mid_point
    
    best_x = (lower_bound + upper_bound) / 2
    best_rmse = calculate_rmse(target_data, input_data * best_x)
    
    return best_x, best_rmse

import time
for i in range(0, 100, 1):
    start_time = time.time()
    x_optimal = optimize_scaling_factor(df_train["Episode_Length_minutes"], df_train["Listening_Time_minutes"], n_searches=i)
    print(i, "\t", x_optimal, "\t",  time.time() - start_time)

0 	 (0.7052511320829684, 13.658310641313289) 	 0.004840850830078125
1 	 (0.8815639151037106, 18.094936064823212) 	 0.0052950382232666016
2 	 (0.7934075235933395, 14.739843669473446) 	 0.004488945007324219
3 	 (0.749329327838154, 13.855502453614294) 	 0.0067043304443359375
4 	 (0.7272902299605613, 13.666725315414762) 	 0.007621049880981445
5 	 (0.7162706810217648, 13.639784114694208) 	 0.008166074752807617
6 	 (0.7107609065523666, 13.64336483446085) 	 0.008589029312133789
7 	 (0.7135157937870658, 13.640152613465325) 	 0.010617971420288086
8 	 (0.7148932374044152, 13.639612842709008) 	 0.010531187057495117
9 	 (0.7155819592130901, 13.639609597426732) 	 0.011373043060302734
10 	 (0.7152375983087527, 13.639588999594187) 	 0.012213945388793945
11 	 (0.7154097787609214, 13.63959374339444) 	 0.01417994499206543
12 	 (0.7153236885348371, 13.639589982713947) 	 0.015892982482910156
13 	 (0.7152806434217949, 13.639589143958897) 	 0.016878843307495117
14 	 (0.7152591208652738, 13.639588984977744) 